# Re-run: recordings and images (RQ4)

From the benchmark repository's committed outputs: `out/speech/*.json` (each audio model's results),
`out/public/video.json` (every video run), `out/images/*.json` (handwritten solutions on three scales),
`language/out/language.json` and `perception/out/perception.json`. The perception bound is the one-sided
95% upper bound on a failure rate with no failures in n recordings, 1 − 0.05^(1/n).

In [1]:
# Finding the published data. This cell is the same in every notebook of this report.
import gzip, json, os, sys
from pathlib import Path

# Importing from the release's code/ folder must not leave __pycache__ folders in it.
sys.dont_write_bytecode = True

RELEASE_ID = "ai-grading-benchmark-2026"   # set when the report is built
DATA_NAME = "per-case-model-outputs.jsonl"


def find_release_file(name=DATA_NAME):
    """Look beside this notebook, in RELEASE_DIR, in the folders above, and in a repository checkout."""
    here = Path.cwd()
    places = [Path(os.environ["RELEASE_DIR"])] if os.environ.get("RELEASE_DIR") else []
    places += [here, *here.parents]
    places += [p / "research/wiki/research/releases" / RELEASE_ID for p in [here, *here.parents]]
    for d in places:
        for n in (name, name + ".gz"):
            if (d / n).is_file():
                return d / n
    raise FileNotFoundError(
        f"Could not find {name} (or {name}.gz). Put it in the same folder as this notebook: download it "
        f"from the report ('Download the data file', beside each notebook) or take it from the release "
        f"directory. Or set the environment variable RELEASE_DIR to the folder that holds it.")


def per_case(runs=None):
    """Yield the per-case records of the published file, optionally only for the named runs."""
    path = find_release_file()
    opener = gzip.open if path.suffix == ".gz" else open
    with opener(path, "rt", encoding="utf-8") as f:
        for line in f:
            r = json.loads(line)
            if r.get("record") == "case" and (runs is None or r["run"] in runs):
                yield r


def code_dir():
    """The release's code package (its code/ folder): in RELEASE_DIR, beside this notebook or in a folder
    above it (as when the notebook sits in the unzipped release's notebooks/ folder), in a folder named
    after the release there, or in a repository checkout."""
    here = Path.cwd()
    places = [Path(os.environ["RELEASE_DIR"])] if os.environ.get("RELEASE_DIR") else []
    for p in [here, *here.parents]:
        places += [p, p / RELEASE_ID, p / "research/wiki/research/releases" / RELEASE_ID]
    for d in places:
        if (d / "code" / "SHA256SUMS").is_file():
            return d / "code"
    raise FileNotFoundError(
        f"Could not find the code package of release {RELEASE_ID} (its code/ folder). Download the release "
        f"from the report ('Download the release', in the sidebar or beside each notebook), unzip it, and "
        f"run this notebook from the unzipped folder or its notebooks/ folder. Or set the environment "
        f"variable RELEASE_DIR to the unzipped {RELEASE_ID} folder.")


def find_code_file(relpath):
    """A file inside the release's code package, by its path there (e.g. studies/.../out/result.json)."""
    path = code_dir() / relpath
    if not path.is_file():
        raise FileNotFoundError(f"The code package at {code_dir()} has no {relpath}: is it release {RELEASE_ID}?")
    return path


def load_code_json(relpath):
    """A committed output of the code package, parsed."""
    return json.loads(find_code_file(relpath).read_text(encoding="utf-8"))


def shown(path):
    """A path as printed: relative to this notebook's folder, so an executed notebook names no machine's folders."""
    return os.path.relpath(path, Path.cwd())


def corpus_file(env_var, name):
    """A public corpus file the reader has downloaded: $env_var/name, or name beside this notebook."""
    for d in ([Path(os.environ[env_var])] if os.environ.get(env_var) else []) + [Path.cwd()]:
        if (d / name).is_file():
            return d / name
    raise FileNotFoundError(f"This notebook also needs {name} from the public corpus. Download the corpus "
                            f"(its location and hash are in the release file), then set {env_var} to its "
                            f"folder or put {name} beside this notebook.")


SOURCE_ENV, SOURCE_NAME = "GBENCH_DIR", "ai-grading-benchmark"   # set when a release's code is another repository


def source_repo():
    """The repository a release's code and committed outputs are published in (a release whose paper cites
    it rather than a copied code package): $SOURCE_ENV, or a checkout named SOURCE_NAME beside this
    notebook or beside a folder above it."""
    here = Path.cwd()
    places = [Path(os.environ[SOURCE_ENV])] if SOURCE_ENV and os.environ.get(SOURCE_ENV) else []
    places += [p / SOURCE_NAME for p in [here, *here.parents]]
    for d in places:
        if d.is_dir() and (d / "out").is_dir():
            return d
    raise FileNotFoundError(
        f"Could not find the {SOURCE_NAME} repository this release's code is published in. Clone it at the "
        f"commit the release file names, then set the environment variable {SOURCE_ENV} to the clone, or "
        f"run this notebook from a folder beside it.")


def load_source_json(relpath):
    """A committed output of that repository, by its path there (cited in the paper as code/<path>)."""
    return json.loads((source_repo() / relpath).read_text(encoding="utf-8"))


print("Source repository:", shown(source_repo()))

Source repository: ../../../../../../ai-grading-benchmark


In [2]:
from decimal import Decimal, ROUND_HALF_UP
from pathlib import Path

def p(x, d=1):
    return f"{Decimal(repr(float(x))).quantize(Decimal(1).scaleb(-d), rounding=ROUND_HALF_UP):.{d}f}"

speech = [load_source_json(f"out/speech/{f.name}") for f in sorted((source_repo() / "out/speech").glob("speech-*.json"))]
sil = [s["validity"]["null_answers"]["by_kind"]["null_silence"]["credited"] for s in speech]
print("silent recordings credited:", [p(100 * x) for x in sil])
assert len(speech) == 7 and sum(float(p(100 * x)) >= 89.7 for x in sil) == 6   # 89.7% to 100%, as printed
lang = load_source_json("language/out/language.json")["models"]
worse = sum(m["gap_high_minus_low"]["ci95"][0] > 0 for m in lang)
print(worse, "of", len(lang), "audio models are worse on low-resource languages")
assert (worse, len(lang)) == (6, 7)

silent recordings credited: ['100.0', '100.0', '100.0', '5.0', '100.0', '96.7', '89.7']
6 of 7 audio models are worse on low-resource languages


In [3]:
runs = [v for v in load_source_json("out/public/video.json") if v["verdict"] != "REFERENCE"]
watches = sum(v["beyond_lo"] is not None and v["beyond_lo"] > 0 for v in runs)
black = sum(v["black_pct"] > 0 for v in runs)
print(len(runs), "video runs;", watches, "grade beyond length;", black, "credit a black screen at least once")
assert (len(runs), watches, black) == (21, 4, 12)

21 video runs; 4 grade beyond length; 12 credit a black screen at least once


In [4]:
raised = 0
names = {f.name.split("-", 1)[1] for f in (source_repo() / "out/images").glob("image*-*.json")}
for n in sorted(names):
    hit = False
    for k in (2, 3, 4):
        v = load_source_json(f"out/images/image{k}-{n}")["validity"]
        hit |= any(v[probe]["ci95"][0] > 0 for probe in ("injection", "forged"))
    raised += hit
print("handwriting: a note on the page raises", raised, "of", len(names), "models on at least one scale")
assert (raised, len(names)) == (12, 15)

handwriting: a note on the page raises 12 of 15 models on at least one scale


In [5]:
pe = load_source_json("perception/out/perception.json")["models"]["google/gemma-4-31b-it"]
n = pe["cannot_see"]["of"]
assert pe["cannot_see"]["n"] == 0
dark = sum(pe["cells"][c]["n"] for c in ("9-10|LIT", "9-10|DARK"))
print("bound overall", p(100 * (1 - 0.05 ** (1 / n)), 2), "% ; darkest group", p(100 * (1 - 0.05 ** (1 / dark))), "%")
assert (p(100 * (1 - 0.05 ** (1 / n)), 2), p(100 * (1 - 0.05 ** (1 / dark)))) == ("0.23", "7.2")

bound overall 0.23 % ; darkest group 7.2 %
